# Lab 46: The Chunking Trick

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 5, *The Memory Hallucination*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-46.ipynb)

**What you will do:** measure your own unchunked and chunked digit span, compare both against Miller's and Cowan's published capacity estimates, pool the class's spans, and see a real language-model tokenizer "chunk" familiar text into far fewer pieces than unfamiliar text.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 46 you tried to recall a 12-digit string with no obvious structure, then tried a second
12-digit string that happened to be three meaningful years — 1969, 2001, 1984 — run together.
The second string is usually much easier, even though it has exactly as many digits, because
it collapses into three "chunks" instead of twelve separate items.

## Record your results

Test your own span twice. First, find the longest *unchunked* digit string (random digits,
no obvious groupings) you can recall perfectly in order. Second, try a 12-digit chunked
string like the book's (or build your own from three memorable years or dates) and record
how many digits you recalled correctly and how many chunks you organised it into.

In [ ]:
# example data: replace with your own results
unchunked_span = 7            # example data: longest unchunked digit string recalled perfectly, in digits
chunked_string_length = 12    # example data: total digits in the chunked string you tried
chunked_digits_recalled = 12  # example data: how many of those digits you recalled correctly
chunked_chunks_used = 3       # example data: how many meaningful chunks you organised the string into

results = pd.DataFrame({
    "measure": ["unchunked span (digits)", "chunked string (digits recalled)", "chunked string (chunks used)"],
    "value": [unchunked_span, chunked_digits_recalled, chunked_chunks_used],
})
display(results)

pd.Series({"unchunked span": unchunked_span, "chunked digits recalled": chunked_digits_recalled}).plot.barh(
    color=["grey", "steelblue"], figsize=(6, 2.5))
plt.xlabel("digits recalled correctly, in order"); plt.show()

## Compare

Miller (1956) put working-memory capacity at roughly 7 ± 2 chunks. Cowan (2001) argues that,
once rehearsal and chunking are prevented, the more fundamental capacity is closer to 4 ± 1
items. Your unchunked span (a plain digit string, which you can still silently rehearse and
lightly chunk) should sit closer to Miller's range; your chunked-string performance shows how
far chunking alone can stretch effective capacity beyond either number.

In [ ]:
print(f"Your unchunked span: {unchunked_span} digits (Miller's range: 5-9; Cowan's stricter range: 3-5)")
if 5 <= unchunked_span <= 9:
    print("Within Miller's classic 7 +/- 2 range.")
elif 3 <= unchunked_span <= 5:
    print("Closer to Cowan's stricter 4 +/- 1 estimate -- perhaps rehearsal or chunking was harder to use here.")
else:
    print("Outside both published ranges -- span tests are noisy for a single trial; try averaging over a few strings.")

print(f"\nChunked string: {chunked_digits_recalled}/{chunked_string_length} digits recalled, "
      f"organised into {chunked_chunks_used} chunks.")
if chunked_chunks_used <= 5 and chunked_digits_recalled > max(9, unchunked_span):
    print("You recalled more digits than your own unchunked span, and more than Miller's upper bound of 9,")
    print("while using a number of CHUNKS that comfortably fits within either 7 +/- 2 or 4 +/- 1 -- exactly")
    print("Miller's point: capacity is limited in chunks, not in raw items.")

## The AI side

The book compares chunking to **tokenisation**: Byte Pair Encoding (BPE) merges frequently
co-occurring character sequences into single tokens, the same way your brain merges a
meaningful digit group into a single chunk (Sennrich, Haddow, & Birch, 2016, *Proceedings of
ACL*). Below, GPT-2's tokenizer — which needs only its vocabulary, not the full model
weights — shows how many tokens it takes to represent common words and familiar numbers
versus a rare or invented word.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

words = ["the", "understanding", "cat", "photosynthesis", "flibbertigibbet", "1969", "196920011984"]
rows = []
for w in words:
    token_ids = tokenizer.encode(w)
    pieces = [tokenizer.decode([t]) for t in token_ids]
    rows.append({"word": w, "n_tokens": len(token_ids), "pieces": pieces})
ai = pd.DataFrame(rows)
display(ai)

long_string_tokens = ai.loc[ai.word == "196920011984", "n_tokens"].iloc[0]
print(f"\n'196920011984' (twelve raw characters) becomes {long_string_tokens} token(s) for GPT-2,")
print("while a genuinely unfamiliar word like 'flibbertigibbet' is split into several small pieces --")
print("the tokenizer's vocabulary has no single 'chunk' for a word it rarely saw during training.")

Common words and familiar digit strings collapse into very few tokens because BPE built its
merge rules from how often those exact sequences appeared in its training text — the same
reason "1969" feels like a single unit to you, but a phone number you have never seen does
not. An invented word like "flibbertigibbet" has no such shortcut in either system: the
tokenizer falls back to smaller pieces, and you would have to remember it letter by letter,
or invent your own private chunk for the occasion.

## Pool the class data

Pool the class: each person adds one row (an anonymous ID, never a name) with their
unchunked span and their chunked-string recall. The example rows below are placeholders so
the cell runs.

In [ ]:
import io
csv_text = """id,unchunked_span,chunked_digits_recalled,chunked_string_length
D01,7,12,12
D02,6,12,12
D03,8,9,12
D04,5,12,12
D05,7,8,12
D06,9,12,12
"""  # example data: replace with your class CSV
class_df = pd.read_csv(io.StringIO(csv_text))
class_df["chunked_prop"] = class_df.chunked_digits_recalled / class_df.chunked_string_length

print(f"Class mean unchunked span: {class_df.unchunked_span.mean():.1f} digits (Miller: 7 +/- 2; Cowan: 4 +/- 1)")
print(f"Class mean chunked-string recall: {class_df.chunked_prop.mean():.0%} of "
      f"{class_df.chunked_string_length.iloc[0]} digits")

boot = [class_df.sample(len(class_df), replace=True).unchunked_span.mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"95% bootstrap interval for mean unchunked span: {lo:.1f}-{hi:.1f} digits (n = {len(class_df)})")

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].hist(class_df.unchunked_span, bins=range(2, 12), color="grey", alpha=0.8)
axes[0].axvspan(5, 9, color="steelblue", alpha=0.15, label="Miller 7 +/- 2")
axes[0].axvspan(3, 5, color="firebrick", alpha=0.15, label="Cowan 4 +/- 1")
axes[0].set_xlabel("unchunked span (digits)"); axes[0].legend(fontsize=7)
axes[1].hist(class_df.chunked_prop, bins=10, color="steelblue", alpha=0.8)
axes[1].set_xlabel("proportion of chunked string recalled"); axes[1].set_xlim(0, 1)
plt.tight_layout(); plt.show()

## Questions to think about

1. Compare your unchunked span with Miller's 7 +/- 2 and Cowan's 4 +/- 1. Which range does it fall closer to, and does that match the book's account of the difference between the two estimates (rehearsal and chunking allowed, versus prevented)?
2. You recalled the 12-digit chunked string using only 3 chunks. If Miller's and Cowan's capacity limits both apply to CHUNKS rather than raw digits, how many raw digits could you in principle hold if each of your 7 +/- 2 (or 4 +/- 1) chunks were, say, five digits long instead of four?
3. In the ai_lab table, compare the number of tokens for "1969" and "196920011984" with the number of tokens for "flibbertigibbet". What does the gap tell you about the relationship between a tokenizer's training data and how well it "chunks" a given string, and how is that similar to the claim that "a non-English speaker cannot chunk 'FBI' into one unit because they do not know the acronym"?
4. A chess master recognises a board position as a single chunk; the text says this is not because their short-term memory is larger. Sketch a version of this Lab's chunking test using chess positions (or another domain you know well, such as song lyrics or sports statistics) instead of digits, and describe what result would support the "same capacity, better chunks" account over a "bigger memory" account.

## Challenge

Take a string of digits you already know as chunks without effort — your own phone number, or
a date of personal significance — and count how many chunks you naturally divide it into.
Then generate a random string of the same total length and time how much longer it takes you
to memorise it to the same standard.

In [ ]:
# Example data: replace with your own comparison.
familiar_string_length = 10        # example data: e.g. a phone number
familiar_string_chunks = 3         # example data: how many chunks you naturally use
familiar_string_learn_seconds = 2  # example data: time to "learn" it (it was already known)

random_string_length = 10          # example data: same length, unfamiliar digits
random_string_learn_seconds = 45   # example data: time needed to memorise it to the same standard

print(f"Familiar string: {familiar_string_length} digits, {familiar_string_chunks} chunks, "
      f"{familiar_string_learn_seconds}s to recall perfectly")
print(f"Random string:   {random_string_length} digits, learned as individual digits, "
      f"{random_string_learn_seconds}s to recall perfectly")
print(f"Ratio: the random string took {random_string_learn_seconds / familiar_string_learn_seconds:.1f}x as long, "
      f"for the same number of digits -- a rough measure of how much chunking is doing for you.")
print("The values above are example data -- replace them with your own timed comparison.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-46.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")